# example_taco.ipynb

# ELLIOT-Pretrain through the TACO reader

[`example.ipynb`](example.ipynb) reads ELLIOT-Pretrain with general-purpose libraries. This notebook reads the same files through the [TACO](https://github.com/OscarPellicer/taco) reader, which knows what every layer is from the dataset's own contract (`ml:contract` in `COLLECTION.json`): its bands, units, scale factor, nodata and frame times. One call returns every layer of a tile, decoded and ready for a model.

Nothing is downloaded in bulk: the reader fetches a part's metadata once and then only the files of the tiles you read.

## Setup

The reader is not yet on PyPI. pip builds it from source, which needs a C++23 compiler, CMake, Ninja, pkg-config, libcurl 7.83 or newer and OpenSSL 3 or newer (on Ubuntu: `apt install build-essential cmake ninja-build pkg-config libcurl4-openssl-dev libssl-dev`; set `CC` and `CXX` if the default compiler is older).

In [ ]:
%pip install -q "taco-eo[ml] @ git+https://github.com/OscarPellicer/taco.git@main#subdirectory=python" matplotlib geopandas

In [ ]:
import os

import geopandas as gpd
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import taco.ml

BASE_URL = os.environ.get("ELLIOT_BASE_URL", "https://data.source.coop/major-tom/elliot-pretrain")
parts = {name: taco.ml.Dataset(f"{BASE_URL}/{name}") for name in ("monotemporal", "monthly", "burst")}
for name, ds in parts.items():
    print(f"{name:13s} {len(ds):>7,} tiles")

## What a tile holds

The contract lists every layer: its kind (a single raster, a series of one raster per acquisition, a mask), its bands and its scaling. The same eight layers exist in every part.

In [ ]:
ds = parts["burst"]
rows = []
for slot in ds.contract.inputs:
    rows.append({
        "layer": slot.name,
        "kind": slot.kind.value,
        "bands": ", ".join(b.common_name or b.polarisation or str(b.index) for b in slot.bands) or "—",
        "scale": slot.scale_factor,
        "classes": ", ".join(slot.classes or []) or "—",
    })
pd.DataFrame(rows)

The sample table has one row per tile: the Major TOM cell, its location, climate, soil, socioeconomic and admin context, and summaries of the added layers (cloud cover, Sentinel-1 coverage, OpenStreetMap feature counts).

In [ ]:
table = ds.table.to_pandas()
table[["majortom:code_10km", "admin:country", "admin:state", "terrain:elevation",
       "cloud:s2_mean_fraction", "s1:frames_with_data", "osm:n_named"]].head()

## One tile, every layer

`ds[i]` decodes every layer of tile `i`. A series has one entry per acquisition; each layer carries its acquisition times.

In [ ]:
INDEX = 120
sample = ds[INDEX]
for name, value in sample.items():
    print(f"{name:14s} {str(value.array.shape):22s} {value.array.dtype}")

In [ ]:
def rgb(stack, scale):
    """True colour from a (bands, H, W) reflectance stack: red, green, blue."""
    return np.clip(stack[[3, 2, 1]].astype(np.float32) * scale / 0.3, 0, 1).transpose(1, 2, 0)


def rgb_l8(stack, scale):
    return np.clip(stack[[3, 2, 1]].astype(np.float32) * scale / 0.3, 0, 1).transpose(1, 2, 0)


s2, l8, s1 = sample["s2"], sample["l8"], sample["s1"]
masks = sample["cloud_mask_s2"].array
s2_scale = ds.contract.inputs[0].scale_factor
l8_scale = next(slot for slot in ds.contract.inputs if slot.name == "l8").scale_factor
dates = pd.to_datetime(list(s2.times), unit="ms").strftime("%Y-%m-%d")

mask_cmap = plt.matplotlib.colors.ListedColormap(["#2a9d8f", "#f4f1de", "#a8dadc", "#3d405b"])
fig, axes = plt.subplots(4, 6, figsize=(24, 16))
for k in range(6):
    axes[0, k].imshow(rgb(s2.array[k], s2_scale))
    axes[0, k].set_title(f"S2 {dates[k]}")
    axes[1, k].imshow(masks[k], cmap=mask_cmap, vmin=0, vmax=3, interpolation="nearest")
    axes[1, k].set_title("cloud mask")
    vv = s1.array[k, 0]
    axes[2, k].imshow(10 * np.log10(np.where(vv > 0, vv, np.nan)), cmap="gray", vmin=-25, vmax=0)
    axes[2, k].set_title(f"S1 VV {str(s1.times[k])[:10]}")
    axes[3, k].imshow(rgb_l8(l8.array[k], l8_scale))
    axes[3, k].set_title("Landsat")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(np.squeeze(sample["dem"].array), cmap="terrain")
axes[0].set_title("Copernicus DEM (m)")
axes[1].imshow(sample["land_cover"].array, cmap="tab20", interpolation="nearest")
axes[1].set_title("ESA WorldCover")
for ax in axes:
    ax.axis("off")
plt.show()

## Per-acquisition metadata

Each acquisition has a metadata row: product, angles, and ERA5 weather at the overpass on the image rows, and cloud fractions on the mask rows. `ds.level(name)` returns a whole level; a row belongs to the tile whose index starts its `internal:relative_path`.

In [ ]:
def tile_rows(level, index):
    rows = ds.level(level).to_pandas()
    rows = rows[rows["internal:relative_path"].str.startswith(f"{index}/")]
    return rows.sort_values("internal:relative_path").reset_index(drop=True)


frames = tile_rows("children/s2", INDEX)
clouds = tile_rows("children/cloud_s2", INDEX)
frames.assign(date=dates, cloud=clouds["ml:cloud_fraction"])[
    ["date", "s2:product_id", "cloud", "ml:era5_temperature_2m_c", "ml:era5_precipitation_mm",
     "ml:era5_wind_speed_10m_kmh"]]

## OpenStreetMap

`osm.parquet` and `admin.parquet` are files of the tile, not model inputs, so they are found through the generic reader: `ds.reader.read(files=...)` gives each tile's location of them.

In [ ]:
import urllib.request


def fetch(location):
    """The bytes at a location the reader gives: a GDAL `/vsicurl/` URL, or a local path."""
    url = location.removeprefix("/vsicurl/")
    if "://" not in url:
        return open(url, "rb")
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return io.BytesIO(urllib.request.urlopen(request).read())


locations = ds.reader.read(files=["osm.parquet", "admin.parquet"]).to_pandas()
where = locations.set_index("majortom:code_10km")
cell = table["majortom:code_10km"].iloc[INDEX]
osm = gpd.read_parquet(fetch(where.loc[cell, "osm.parquet::location"]))
admin = gpd.read_parquet(fetch(where.loc[cell, "admin.parquet::location"]))
print(admin[["name", "admin_level"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 8))
osm.plot(ax=ax, column="category", legend=True, linewidth=0.8, markersize=4,
         legend_kwds={"loc": "upper left", "bbox_to_anchor": (1, 1)})
ax.set_title(f"OpenStreetMap, {cell}")
ax.set_axis_off()
plt.show()

## A seasonal series

The monthly part has one acquisition per calendar month, not necessarily from the same year: a multi-year climatology. `read` decodes only the layers asked for.

In [ ]:
monthly = parts["monthly"]
series = monthly.read(8000, slots=["s2", "cloud_mask_s2"])
months = pd.to_datetime(list(series["s2"].times), unit="ms")

fig, axes = plt.subplots(2, 12, figsize=(36, 6))
for k in range(12):
    axes[0, k].imshow(rgb(series["s2"].array[k], s2_scale))
    axes[0, k].set_title(months[k].strftime("%b %Y"))
    axes[1, k].imshow(series["cloud_mask_s2"].array[k], cmap=mask_cmap, vmin=0, vmax=3,
                      interpolation="nearest")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Training on it

`taco.ml.Dataset` is indexable, so it can be wrapped in a PyTorch `Dataset` directly. For vision-language tasks built from these layers (captions, grounding, grids, phenology and change questions, with scorers), see [majortom-elliot-tasks](https://github.com/elliot-project/majortom-elliot-tasks).

In [ ]:
%pip install -q torch

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset


class Tiles(Dataset):
    def __init__(self, ds, slots):
        self.ds, self.slots = ds, slots

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, index):
        sample = self.ds.read(index, slots=self.slots)
        return {name: torch.from_numpy(np.asarray(value.array, dtype=np.float32))
                for name, value in sample.items()}


loader = DataLoader(Tiles(parts["burst"], ["s2", "s1", "cloud_mask_s2"]), batch_size=2)
batch = next(iter(loader))
{name: tuple(value.shape) for name, value in batch.items()}